## Building a GPT

Companion notebook to the [Zero To Hero](https://karpathy.ai/zero-to-hero.html) video on GPT.

In [1]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-04-29 10:35:48--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.04s   

2026-04-29 10:35:48 (29.2 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

### Full finished code, for reference

You may want to refer directly to the git repo instead though.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 5 # what is the maximum context length for predictions?
max_iters = 200 # kitne iterations tk code chlega
eval_interval = 100# kitni. der baad output print hota rhega
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64  # dimensios of the embedding layer
n_head = 4    # number of heads of attention
n_layer = 4   # attention for 4 layers
dropout = 0.0
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers

# our tokenizer in this case is character based abd very very simple as explained below

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
num = 1
def get_batch(split):
    global num
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))  # randomly get Batch_number indices
    x = torch.stack([data[i:i+block_size] for i in ix])       # generate batches using those indices of block_size size
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])    # actual output


    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) # kind of just a variable in pytorch

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

      # for parallelly heads, each head will get the same input (B,T,N_Embed)
        B,T,N_Embed = x.shape

        k = self.key(x)   # B,T, N_Embed  * (N_Embed , Head_size) ==== (B,T, Head_Size)
        q = self.query(x) # B,T, N_Embed  * (N_Embed , Head_size) ==== (B,T, Head_Size)

        # compute attention scores ("affinities")

        wei = q @ k.transpose(-2,-1) * head_size**-0.5 # (B, T, N_Embed) @ (B, N_Embed, T) -> (B, T, T) batch x block_size x block_size

        # basically ye affinities aa gyi ek  batch k har token ki apne batch k baaki tokens k saath

        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T).  ye kia because we dont want it to see aagey walie tokens


        wei = F.softmax(wei, dim=-1) # (B, T, T)      # previous weights k saath usko weight dene k lie

        wei = self.dropout(wei) # random dropdouts make it better

        # perform the weighted aggregation of the values
        v = self.value(x)  # B,T, N_Embed  * (N_Embed , Head_size) ==== (B,T, Head_Size)

        out = wei @ v # (B, T, T) @ (B, T, Head_Size) -> (B, T, Head_Size)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) # we get B,T,Head_Size from each Head, and then we concatenate then to get B,T,N_Embed
        # across the last dimension, bas saari matrices ko chipka de rhe hain. toh head_size*num_heads bn jayega, which is n_embeddings

        projection = self.proj(out) # this is done so that the heads , which are for now just glued, can also talk to each other

        return self.dropout(projection)

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd) # LAyer norm is the equalizer. Baar baar sum k baad bada chota hota jata hai sb isse hi fix hota hai

    def forward(self, x):
        x1 = self.ln1(x) # (B, T, N_Embed) Layer norm does not change the size just fixes and smoothes things

        x = x + self.sa(x1) #(B , T, N_Embed) after the complete multi head attention stuff


        x2 = self.ln2(x) # layer norm  (B , T, N_Embed)

        x = x + self.ffwd(x2) # the OG Feed forward . The conversions are fairly simple here

        return x


        # x = x + self.sa(self.ln1(x))     # residual layer norm is applied before the input goes into self attention and
        # x = x + self.ffwd(self.ln2(x))     # rresidual later norm is applied before it goes to feed forward neural network also
        # return x

        # LayerNorm -> Attention -> LayerNorm -> Feed Forward -> LayerNorm -> Attention -> LayerNorm -> Feedforward

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd) # vocab size transformed to n_embd dimension


        self.position_embedding_table = nn.Embedding(block_size, n_embd) # for each token in our block, its positional encoding is also stored, also transformning to n_embd

        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)]) # n_layer is 4, so there are 4 attention layers

        self.ln_f = nn.LayerNorm(n_embd) # layer norm that is done in the end

        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        # idx is the x parameter, the input
        # target is the y parameter, the output
        # when we do model(x,y), it calls model.forward() directly

        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,N_Embed)
        # so basically from the token embedding table, for each token in idx, we will be getting the corresponding n_embd vector/row making token_embed as (B,T, N_Embed)

        # since there are like 65 different tokens possible(vocab_size, the Embedding is 65 x n_embd)

        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,N_Embed.. torch.arange(T) gives 0,1,2,3.... T)

        x = tok_emb + pos_emb # (B,T,N_Embed) positional embedding is batched/copied automatically to get this

        x = self.blocks(x) # (B,T,N_Embed) # from the attention and FF blocks according to architecture

        x = self.ln_f(x) # (B,T,N_Embed)

        logits = self.lm_head(x) # (B,T, N_Embed ) * (N_Embd , Vocab_size) = (B , T, Vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, V_size = logits.shape
            logits = logits.view(B*T, V_size) # because pytorch needs without batching to calculate outputs
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            print(idx_cond)
            # get the predictions
            logits, loss = self(idx_cond) # since targets is none, so no loss
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes Only C. We take the logits of only the last one
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # Only C since there is no batch
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (T+1), no batch here
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
# for name, p in m.named_parameters():
#     print(f"{name:45} {tuple(p.shape)} {p.numel():,}")
#print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')


# calculating the number of params manually down below

head_size = n_embd // n_head

blockParams  = n_embd* n_embd + n_embd  +  n_embd* n_embd * 4 + 4*n_embd + n_embd*4*n_embd + n_embd+  2*n_embd + 2*n_embd + 4*(3*n_embd*head_size)

calculatedParams =   vocab_size * n_embd + block_size * n_embd + n_layer*(blockParams) + 2*n_embd + n_embd * vocab_size + vocab_size

# paramateres in nn.linear(a,b) = a*b + b (last b for bias)
# parameteres in nn.LayerNorm(b) are 2*b
# parameteres in nn.Embedding(a,b) are a*b


print(' num2 ', calculatedParams)


# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

        # bs print krne k lie training and validation dono pr loss nikal rhe hain

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


 num2  208001
step 0: train loss 4.4609, val loss 4.4453
step 100: train loss 3.0608, val loss 3.0559
step 199: train loss 2.7858, val loss 2.7938
tensor([[0]])
tensor([[ 0, 28]])
tensor([[ 0, 28, 46]])
tensor([[ 0, 28, 46, 47]])
tensor([[ 0, 28, 46, 47, 12]])
tensor([[28, 46, 47, 12,  1]])
tensor([[46, 47, 12,  1, 61]])
tensor([[47, 12,  1, 61, 43]])
tensor([[12,  1, 61, 43,  1]])
tensor([[ 1, 61, 43,  1, 58]])
tensor([[61, 43,  1, 58, 46]])
tensor([[43,  1, 58, 46, 43]])
tensor([[ 1, 58, 46, 43, 52]])
tensor([[58, 46, 43, 52, 42]])
tensor([[46, 43, 52, 42, 54]])
tensor([[43, 52, 42, 54,  1]])
tensor([[52, 42, 54,  1, 44]])
tensor([[42, 54,  1, 44, 10]])
tensor([[54,  1, 44, 10, 53]])
tensor([[ 1, 44, 10, 53, 12]])
tensor([[44, 10, 53, 12, 21]])
tensor([[10, 53, 12, 21,  0]])
tensor([[53, 12, 21,  0, 30]])
tensor([[12, 21,  0, 30, 20]])
tensor([[21,  0, 30, 20,  0]])
tensor([[ 0, 30, 20,  0, 24]])
tensor([[30, 20,  0, 24,  7]])
tensor([[20,  0, 24,  7, 25]])
tensor([[ 0, 24,  7, 25,  

KeyboardInterrupt: 